# 04 — DistilBERT Prosodic Boundary Classifier (Multi-corpus)

Fine-tunes `distilbert-base-uncased` on prosodic boundary labels from one or more corpora.

**Task:** Multi-task token classification — boundary (`b`), intonation type (`i`), break index (`x`).

**Baseline to beat:** PSST text-only GPT-Neo F1 = 0.77

**Fixed test set:** SBC001–005 (gold conversational, held out from all training).

Disclaimer: Code was largely generated with the help of Claude Sonnet 4.6 (Anthropic, 2026). Prompts, code tweaks and verification by me.

---

## How to use this notebook

Run cells **top to bottom**. Cells you will edit regularly:

| Cell | Purpose |
|---|---|
| **Cell 1** | Paths, hyperparameter defaults, corpus load flags |
| **Cell 4** | Corpus loading (runs automatically based on flags) |
| **Cell 14** | Run queue — define which datasets + overrides per run |

Everything else runs unchanged between experiments.

---

## Corpus registry

Corpora are loaded in Cell 4 into a `CORPUS_REGISTRY` dict, then referenced by name in the run queue:

| Key | Corpus | Notes |
|---|---|---|
| `"libri"` | LibriTTS clean-100 + clean-360 | Silver-standard, read speech, ~145k samples |
| `"sbc"` | SBCSAE SBC006–SBC060 | Gold-standard, conversational (SBC001–005 are test-only) |
| `"bu"` | BU Radio News Corpus | Gold-standard, broadcast news, ~426 samples |

Add future corpora by adding a flag + load snippet in Cell 4 and a new key to `CORPUS_REGISTRY`.

---

## Output folder structure

```
runs/
└── {run_notes}/
    ├── checkpoint/                    ← model weights + tokenizer
    ├── {run_id}_hparams.json
    ├── {run_id}_test_predictions.json
    ├── {run_id}_curves.png
    └── {run_id}_confusion_matrix.png
```


---
## Section 1 · Configuration

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — CONFIGURATION                                      ║
# ║  Edit paths, hyperparameter defaults, and corpus flags here. ║
# ╚══════════════════════════════════════════════════════════════╝
import os, json, hashlib
from datetime import datetime

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/Capstone/project"
RUNS_ROOT  = f"{DRIVE_ROOT}/runs"

# Corpus label roots — add new corpora here
BATCHED_ROOT_LIBRI_100 = f"{DRIVE_ROOT}/labels/clean-100"
BATCHED_ROOT_LIBRI_360 = f"{DRIVE_ROOT}/labels/clean-360"
BATCHED_ROOT_SBC       = f"{DRIVE_ROOT}/labels/sbcsae"
BATCHED_ROOT_BU        = f"{DRIVE_ROOT}/labels/bu"
# BATCHED_ROOT_NEWCORPUS = f"{DRIVE_ROOT}/labels/newcorpus"

# ── Corpus load flags ─────────────────────────────────────────────────────────
# Set False to skip loading a corpus (saves time if you won't use it this session).
# A run that requests a skipped corpus will raise a KeyError.
LOAD_LIBRI = True   # ~145k samples — slow to load
LOAD_SBC   = True   # ~4k samples
LOAD_BU    = True   # ~426 samples
# LOAD_NEWCORPUS = True

# SBC test holdout IDs — these are NEVER added to the registry or any training set
SBC_TEST_IDS = set(f"SBC{str(n).zfill(3)}" for n in range(1, 6))  # SBC001–SBC005

# ── Data split (train/val only — test is always SBC001–005) ───────────────────
SPLIT_SEED = 42
TRAIN_FRAC = 0.80
VAL_FRAC   = 0.10

# ── Text pre-processing ───────────────────────────────────────────────────────
STRIP_PUNCTUATION = False
EXTRA_FEATURES    = []

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 10
WARMUP_STEPS  = 0
WEIGHT_DECAY  = 0.01

# ── Class-imbalance strategy (boundary head only) ─────────────────────────────
IMBALANCE_STRATEGY   = "none"
BOUNDARY_LOSS_WEIGHT = 5.0

# ── Multi-task loss weights ───────────────────────────────────────────────────
BOUNDARY_TASK_WEIGHT   = 1.0
INTONATION_TASK_WEIGHT = 1.0
BREAK_IDX_TASK_WEIGHT  = 1.0

# ── Run metadata ──────────────────────────────────────────────────────────────
RUN_NOTES = ""

print("✓ Configuration loaded.")
print(f"  Corpus flags:  LIBRI={LOAD_LIBRI}  SBC={LOAD_SBC}  BU={LOAD_BU}")
print(f"  SBC test IDs:  {sorted(SBC_TEST_IDS)}")


---
## Section 2 · Environment Setup

Run once per Colab session. Safe to re-run — all steps are idempotent.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Mount Drive + install packages                     ║
# ╚══════════════════════════════════════════════════════════════╝
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=True)
os.makedirs(RUNS_ROOT, exist_ok=True)

!pip install -q transformers datasets scikit-learn matplotlib

import torch
print(f"\n✓ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("⚠  No GPU — training will be slow.")
    print("   Runtime → Change runtime type → T4 GPU")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   Device: {device}")


---
## Section 3 · Imports

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Imports                                            ║
# ╚══════════════════════════════════════════════════════════════╝
import os, json, string, random, shutil, traceback, gc, hashlib, subprocess
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from transformers import (
    DistilBertTokenizerFast,
    DistilBertModel,
    DistilBertPreTrainedModel,
    DistilBertConfig,
    get_linear_schedule_with_warmup,
)
from tqdm.notebook import tqdm, tqdm as tqdm_nb

class _NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        return super().default(obj)

torch.manual_seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)
random.seed(SPLIT_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SPLIT_SEED)

print("✓ Imports complete.")


---
## Section 4 · Load Corpora

Loads each corpus based on the flags in Cell 1. Splits SBC into train (`sbc`) and fixed test holdout (`samples_sbc_test`) at load time.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Load corpora                                       ║
# ║                                                              ║
# ║  Loads each corpus based on flags set in Cell 1.             ║
# ║  Builds CORPUS_REGISTRY (training data only) and             ║
# ║  samples_sbc_test (fixed holdout — never in registry).       ║
# ║                                                              ║
# ║  To add a future corpus:                                     ║
# ║    1. Add LOAD_NEWCORPUS flag + BATCHED_ROOT_NEWCORPUS in C1 ║
# ║    2. Add a load block below (copy any existing block)       ║
# ║    3. Add key → samples to CORPUS_REGISTRY at the bottom     ║
# ╚══════════════════════════════════════════════════════════════╝

def load_label_files_batched(batched_root, label=""):
    """
    Load all samples from batch_*.json files in batched_root.

    Batch file layout (produced by annotation_pipeline_*.ipynb):
        batch_0000.json  ← dict keyed by sample_id, each value has "b", "i", "x"
        ...

    Returns
    -------
    samples : list[dict]  — keys: sample_id, tokens, boundary_labels,
                            intonation_labels, break_idx_labels
    meta    : dict        — empty placeholder (meta.json not bundled)
    """
    batch_files = sorted(
        f for f in os.listdir(batched_root)
        if f.startswith("batch_") and f.endswith(".json")
    )
    if not batch_files:
        raise FileNotFoundError(
            f"No batch files found in {batched_root}.\n"
            "Check path or run the relevant annotation pipeline first."
        )
    tag = f" [{label}]" if label else ""
    print(f"Found {len(batch_files)} batch files in {batched_root}{tag}.")

    samples, n_skipped = [], 0
    for fname in tqdm(batch_files, desc=f"Loading{tag}", unit="batch"):
        with open(os.path.join(batched_root, fname)) as f:
            batch = json.load(f)
            for sid, data in batch.items():
                b, i, x = data["b"], data["i"], data["x"]
                n = len(b["tokens"])
                x_labels = x["labels"] if x is not None else [""] * n
                if not (len(b["consensus"]) == len(i["labels"]) == len(x_labels) == n):
                    print(f"  ⚠ {sid}: mismatched label lengths — skipping.")
                    n_skipped += 1
                    continue
                samples.append({
                    "sample_id":         sid,
                    "tokens":            b["tokens"],
                    "boundary_labels":   b["consensus"],
                    "intonation_labels": i["labels"],
                    "break_idx_labels":  x_labels,
                })

    print(f"  ✓ {len(samples):,} samples loaded  ({n_skipped} skipped).")
    if samples:
        total_tokens     = sum(len(s["tokens"])          for s in samples)
        total_boundaries = sum(sum(s["boundary_labels"]) for s in samples)
        pos_rate         = total_boundaries / max(total_tokens, 1)
        ratio            = (1 - pos_rate) / max(pos_rate, 1e-9)
        print(f"  Total words:     {total_tokens:,}")
        print(f"  Boundary rate:   {100*pos_rate:.1f}%  (ratio ≈ {ratio:.1f}:1)")
        print(f"  → Suggested BOUNDARY_LOSS_WEIGHT: ~{round(ratio)}.0")
    return samples, {}


# ── CORPUS_REGISTRY (training data only) ─────────────────────────────────────
# Maps name → sample list. Only loaded corpora appear here.
CORPUS_REGISTRY = {}

# ── LibriTTS ──────────────────────────────────────────────────────────────────
if LOAD_LIBRI:
    _samples_100, _meta_100 = load_label_files_batched(BATCHED_ROOT_LIBRI_100, "libri-100")
    _samples_360, _meta_360 = load_label_files_batched(BATCHED_ROOT_LIBRI_360, "libri-360")
    samples_libri = _samples_100 + _samples_360
    print(f"  LibriTTS total: {len(samples_libri):,} samples")
    CORPUS_REGISTRY["libri"] = samples_libri
else:
    print("  ⚠  LOAD_LIBRI=False — 'libri' not in registry.")

# ── SBCSAE ────────────────────────────────────────────────────────────────────
if LOAD_SBC:
    _all_sbc, _ = load_label_files_batched(BATCHED_ROOT_SBC, "sbcsae")
    # Split at load time: SBC001–005 → fixed test holdout (never used for training)
    samples_sbc_test  = [s for s in _all_sbc
                         if s["sample_id"][:6] in SBC_TEST_IDS]
    samples_sbc_train = [s for s in _all_sbc
                         if s["sample_id"][:6] not in SBC_TEST_IDS]
    print(f"  SBC train: {len(samples_sbc_train):,}  |  SBC test (holdout): {len(samples_sbc_test):,}")
    CORPUS_REGISTRY["sbc"] = samples_sbc_train
else:
    print("  ⚠  LOAD_SBC=False — 'sbc' not in registry.")
    samples_sbc_test = []
    print("  ⚠  samples_sbc_test is empty — test set will be a random split.")

# ── BU Radio News ─────────────────────────────────────────────────────────────
if LOAD_BU:
    samples_bu, _ = load_label_files_batched(BATCHED_ROOT_BU, "bu")
    CORPUS_REGISTRY["bu"] = samples_bu
else:
    print("  ⚠  LOAD_BU=False — 'bu' not in registry.")

# ── ADD FUTURE CORPORA HERE ───────────────────────────────────────────────────
# if LOAD_NEWCORPUS:
#     samples_newcorpus, _ = load_label_files_batched(BATCHED_ROOT_NEWCORPUS, "newcorpus")
#     CORPUS_REGISTRY["newcorpus"] = samples_newcorpus
# else:
#     print("  ⚠  LOAD_NEWCORPUS=False — 'newcorpus' not in registry.")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n✓ CORPUS_REGISTRY keys:", list(CORPUS_REGISTRY.keys()))
print(f"  Fixed test set (SBC001–005): {len(samples_sbc_test):,} samples")


---
## Section 5 · Dataset

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — ProsodyDataset + collate_fn                        ║
# ╚══════════════════════════════════════════════════════════════╝

PUNCT_SET = set(string.punctuation)

def is_punct_only(word):
    return bool(word) and all(c in PUNCT_SET for c in word)


class ProsodyDataset(Dataset):
    """
    Token-classification dataset for prosodic boundary detection.

    Label alignment rules
    ─────────────────────
    All three heads share the same rule: first sub-word of word i receives
    the word's label; continuation sub-words and special tokens get -100.

    Boundary   : 0 / 1  — all positions supervised
    Intonation : rising(1)→0 | falling(2)→1 | level(3)→2
                 none(0) and unclear(4) → -100  (masked)
    Break index: "3"→0 | "4"→1
                 "" and any missing value → -100 (masked)
                 ("" means unannotated, not an affirmative non-boundary)
    """

    SPK_CHANGE_TOKEN = "/"

    def __init__(self, samples, tokenizer, max_length=128,
                 strip_punctuation=False, extra_features=None):
        self.tokenizer         = tokenizer
        self.max_length        = max_length
        self.strip_punctuation = strip_punctuation
        self.extra_features    = extra_features or []
        self.items             = [self._process(s) for s in samples]

    def _process(self, sample):
        words    = list(sample["tokens"])
        b_labels = list(sample["boundary_labels"])
        i_labels = list(sample.get("intonation_labels", []))
        x_labels = list(sample.get("break_idx_labels",  []))

        if self.strip_punctuation:
            quads = [
                (w, b, i, x)
                for w, b, i, x in zip(words, b_labels, i_labels, x_labels)
                if not is_punct_only(w) or w == self.SPK_CHANGE_TOKEN
            ]
            if quads:
                words, b_labels, i_labels, x_labels = map(list, zip(*quads))
            else:
                words, b_labels, i_labels, x_labels = [], [], [], []

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        word_ids = encoding.word_ids(batch_index=0)

        def align_labels(label_list, default=-100):
            aligned, prev = [], None
            for word_i in word_ids:
                if word_i is None:
                    aligned.append(default)
                elif word_i != prev:
                    aligned.append(label_list[word_i] if word_i < len(label_list) else default)
                else:
                    aligned.append(default)
                prev = word_i
            return aligned

        aligned_boundary = align_labels(b_labels, default=-100)

        aligned_spk_mask = []
        prev = None
        for word_i in word_ids:
            if word_i is None:
                aligned_spk_mask.append(False)
            elif word_i != prev:
                aligned_spk_mask.append(words[word_i] == self.SPK_CHANGE_TOKEN)
            else:
                aligned_spk_mask.append(False)
            prev = word_i

        _INTONATION_MAP = {1: 0, 2: 1, 3: 2}
        aligned_intonation = []
        prev = None
        for word_i in word_ids:
            if word_i is None:
                aligned_intonation.append(-100)
            elif word_i != prev:
                raw = i_labels[word_i] if word_i < len(i_labels) else -1
                aligned_intonation.append(_INTONATION_MAP.get(raw, -100))
            else:
                aligned_intonation.append(-100)
            prev = word_i

        _BREAK_IDX_MAP = {"3": 0, "4": 1}
        aligned_break_idx = []
        prev = None
        for word_i in word_ids:
            if word_i is None:
                aligned_break_idx.append(-100)
            elif word_i != prev:
                raw = x_labels[word_i] if word_i < len(x_labels) else ""
                aligned_break_idx.append(_BREAK_IDX_MAP.get(raw, -100))
            else:
                aligned_break_idx.append(-100)
            prev = word_i

        extra = {}
        # if "pos" in self.extra_features:
        #     extra["pos_ids"] = build_pos_ids(words, encoding, self.max_length)

        return {
            "sample_id":         sample["sample_id"],
            "tokens":            words,
            "input_ids":         encoding["input_ids"].squeeze(0),
            "attention_mask":    encoding["attention_mask"].squeeze(0),
            "spk_mask":          torch.tensor(aligned_spk_mask, dtype=torch.bool),
            "labels":            torch.tensor(aligned_boundary,   dtype=torch.long),
            "intonation_labels": torch.tensor(aligned_intonation, dtype=torch.long),
            "break_idx_labels":  torch.tensor(aligned_break_idx,  dtype=torch.long),
            **extra,
        }

    def __len__(self):  return len(self.items)
    def __getitem__(self, idx): return self.items[idx]


def prosody_collate_fn(batch):
    out = {
        "input_ids":         torch.stack([b["input_ids"]         for b in batch]),
        "attention_mask":    torch.stack([b["attention_mask"]    for b in batch]),
        "spk_mask":          torch.stack([b["spk_mask"]          for b in batch]),
        "labels":            torch.stack([b["labels"]            for b in batch]),
        "intonation_labels": torch.stack([b["intonation_labels"] for b in batch]),
        "break_idx_labels":  torch.stack([b["break_idx_labels"]  for b in batch]),
        "sample_ids":        [b["sample_id"] for b in batch],
        "tokens":            [b["tokens"]    for b in batch],
    }
    # if "pos_ids" in batch[0]:
    #     out["pos_ids"] = torch.stack([b["pos_ids"] for b in batch])
    return out


print("✓ ProsodyDataset and prosody_collate_fn defined.")


---
## Section 7 · Model

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — ProsodyBoundaryModel                               ║
# ╚══════════════════════════════════════════════════════════════╝

class ProsodyBoundaryModel(DistilBertPreTrainedModel):
    """
    DistilBERT encoder with three classification heads:
        boundary_head    Linear(H → 2)   all positions
        intonation_head  Linear(H → 3)   rising / falling / level
        break_idx_head   Linear(H → 2)   index-3 / index-4
    """

    def __init__(self, config):
        super().__init__(config)
        self.distilbert      = DistilBertModel(config)
        self.dropout         = nn.Dropout(config.seq_classif_dropout)
        self.boundary_head   = nn.Linear(config.hidden_size, 2)
        self.intonation_head = nn.Linear(config.hidden_size, 3)
        self.break_idx_head  = nn.Linear(config.hidden_size, 2)
        self.post_init()

    def forward(self, input_ids, attention_mask,
                labels=None, intonation_labels=None, break_idx_labels=None,
                loss_weights=None):
        outputs = self.distilbert(input_ids=input_ids,
                                  attention_mask=attention_mask)
        seq_out = self.dropout(outputs.last_hidden_state)

        boundary_logits   = self.boundary_head(seq_out)
        intonation_logits = self.intonation_head(seq_out)
        break_idx_logits  = self.break_idx_head(seq_out)

        losses = {}
        if labels is not None:
            losses["boundary"] = nn.CrossEntropyLoss(
                weight=loss_weights, ignore_index=-100)(
                boundary_logits.view(-1, 2), labels.view(-1))
        if intonation_labels is not None and (intonation_labels != -100).any():
            losses["intonation"] = nn.CrossEntropyLoss(ignore_index=-100)(
                intonation_logits.view(-1, 3), intonation_labels.view(-1))
        if break_idx_labels is not None and (break_idx_labels != -100).any():
            losses["break_idx"] = nn.CrossEntropyLoss(ignore_index=-100)(
                break_idx_logits.view(-1, 2), break_idx_labels.view(-1))

        out = {
            "boundary_logits":   boundary_logits,
            "intonation_logits": intonation_logits,
            "break_idx_logits":  break_idx_logits,
        }
        if losses:
            out["losses"] = losses
        return out

@classmethod
def _can_set_experts_implementation(cls):
    return False
ProsodyBoundaryModel._can_set_experts_implementation = _can_set_experts_implementation

def build_model(model_name=MODEL_NAME):
    cfg = DistilBertConfig.from_pretrained(model_name,
                                           num_labels=2,
                                           seq_classif_dropout=0.2)
    model = ProsodyBoundaryModel.from_pretrained(model_name, config=cfg,
                                                  ignore_mismatched_sizes=True,
                                                  _fast_init=False)
    model.gradient_checkpointing_enable()
    return model.to(device)

print("✓ ProsodyBoundaryModel defined.")


---
## Section 8 · Metrics

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Metrics helpers                                    ║
# ╚══════════════════════════════════════════════════════════════╝

def flatten_predictions(all_logits, all_labels, all_spk_masks=None):
    flat_preds, flat_labels = [], []
    for idx, (logits_batch, labels_batch) in enumerate(zip(all_logits, all_labels)):
        preds_batch = logits_batch.argmax(dim=-1)
        spk_batch   = all_spk_masks[idx] if all_spk_masks else None
        for b_i, (preds_seq, labels_seq) in enumerate(zip(preds_batch, labels_batch)):
            mask = labels_seq != -100
            if spk_batch is not None:
                mask = mask & ~spk_batch[b_i].to(mask.device)
            flat_preds.extend(preds_seq[mask].cpu().tolist())
            flat_labels.extend(labels_seq[mask].cpu().tolist())
    return np.array(flat_preds), np.array(flat_labels)

def compute_metrics(flat_preds, flat_labels):
    return {
        "f1":        f1_score(flat_labels,        flat_preds, pos_label=1, zero_division=0),
        "precision": precision_score(flat_labels, flat_preds, pos_label=1, zero_division=0),
        "recall":    recall_score(flat_labels,    flat_preds, pos_label=1, zero_division=0),
    }

def compute_multiclass_metrics(flat_preds, flat_labels):
    return {
        "f1":        f1_score(flat_labels,        flat_preds, average="macro", zero_division=0),
        "precision": precision_score(flat_labels, flat_preds, average="macro", zero_division=0),
        "recall":    recall_score(flat_labels,    flat_preds, average="macro", zero_division=0),
    }

print("✓ Metrics helpers defined.")


---
## Section 12 · Multi-run harness

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Multi-run harness                                 ║
# ║  Define run_experiment() — called by Cell 14 for each run.   ║
# ╚══════════════════════════════════════════════════════════════╝

def run_experiment(overrides: dict, datasets: list[str]):
    """
    Train, evaluate, and save one DistilBERT run.

    Parameters
    ----------
    overrides : dict
        Any subset of Cell-1 config keys. Unspecified keys fall back to
        Cell-1 globals.
    datasets : list[str]
        Names of corpora to combine for train/val. Each must be a key in
        CORPUS_REGISTRY. Example: ["libri", "sbc", "bu"].
        BU (and any corpus without SBC IDs) goes entirely into train/val —
        it is never confused with the test holdout.

    Test set
    --------
    Always samples_sbc_test (SBC001–005). If LOAD_SBC=False, falls back
    to a random 10% slice of the training pool with a warning.
    """

    # ── 1. Resolve config ─────────────────────────────────────────────────────
    run_cfg = dict(
        SPLIT_SEED           = SPLIT_SEED,
        TRAIN_FRAC           = TRAIN_FRAC,
        VAL_FRAC             = VAL_FRAC,
        STRIP_PUNCTUATION    = STRIP_PUNCTUATION,
        EXTRA_FEATURES       = EXTRA_FEATURES,
        MODEL_NAME           = MODEL_NAME,
        MAX_LENGTH           = MAX_LENGTH,
        BATCH_SIZE           = BATCH_SIZE,
        LEARNING_RATE        = LEARNING_RATE,
        NUM_EPOCHS           = NUM_EPOCHS,
        WARMUP_STEPS         = WARMUP_STEPS,
        WEIGHT_DECAY         = WEIGHT_DECAY,
        IMBALANCE_STRATEGY   = IMBALANCE_STRATEGY,
        BOUNDARY_LOSS_WEIGHT = BOUNDARY_LOSS_WEIGHT,
        BOUNDARY_TASK_WEIGHT     = BOUNDARY_TASK_WEIGHT,
        INTONATION_TASK_WEIGHT   = INTONATION_TASK_WEIGHT,
        BREAK_IDX_TASK_WEIGHT    = BREAK_IDX_TASK_WEIGHT,
        RUN_NOTES            = RUN_NOTES,
    )
    run_cfg.update(overrides)

    _SPLIT_SEED  = run_cfg["SPLIT_SEED"]
    _TRAIN_FRAC  = run_cfg["TRAIN_FRAC"]
    _VAL_FRAC    = run_cfg["VAL_FRAC"]
    _STRIP_PUNCT = run_cfg["STRIP_PUNCTUATION"]
    _EXTRA_FEATURES = run_cfg["EXTRA_FEATURES"]
    _MODEL_NAME  = run_cfg["MODEL_NAME"]
    _MAX_LENGTH  = run_cfg["MAX_LENGTH"]
    _BATCH_SIZE  = run_cfg["BATCH_SIZE"]
    _LR          = run_cfg["LEARNING_RATE"]
    _EPOCHS      = run_cfg["NUM_EPOCHS"]
    _WARMUP      = run_cfg["WARMUP_STEPS"]
    _WD          = run_cfg["WEIGHT_DECAY"]
    _IMBALANCE   = run_cfg["IMBALANCE_STRATEGY"]
    _BLW         = run_cfg["BOUNDARY_LOSS_WEIGHT"]
    _BTW         = run_cfg["BOUNDARY_TASK_WEIGHT"]
    _ITW         = run_cfg["INTONATION_TASK_WEIGHT"]
    _XTW         = run_cfg["BREAK_IDX_TASK_WEIGHT"]
    _NOTES       = run_cfg["RUN_NOTES"]

    _cfg_hash = hashlib.md5(
        json.dumps({k: v for k, v in run_cfg.items() if k != "RUN_NOTES"},
                   sort_keys=True).encode()
    ).hexdigest()[:8]
    run_id  = _NOTES if _NOTES else f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{_cfg_hash}"
    run_dir = f"{RUNS_ROOT}/{run_id}"

    print("\n" + "█" * 70)
    print(f"  RUN: {run_id}")
    print(f"  Datasets:    {datasets}")
    print(f"  Strip punct: {_STRIP_PUNCT}")
    print(f"  Imbalance:   {_IMBALANCE}  (weight={_BLW})")
    print(f"  LR / Epochs: {_LR} / {_EPOCHS}")
    print("█" * 70)

    # ── 2. Resolve training pool from registry ────────────────────────────────
    missing = [k for k in datasets if k not in CORPUS_REGISTRY]
    if missing:
        raise KeyError(
            f"Dataset(s) {missing} not found in CORPUS_REGISTRY.\n"
            f"Available: {list(CORPUS_REGISTRY.keys())}\n"
            f"Check LOAD_* flags in Cell 1."
        )
    _train_pool = []
    for key in datasets:
        _train_pool.extend(CORPUS_REGISTRY[key])
    print(f"  Training pool: {len(_train_pool):,} samples  ({' + '.join(f'{k}={len(CORPUS_REGISTRY[k]):,}' for k in datasets)})")

    # ── 3. Test set ───────────────────────────────────────────────────────────
    if samples_sbc_test:
        _test_items = samples_sbc_test
        print(f"  Test set: SBC001–005 fixed holdout ({len(_test_items):,} samples)")
        _split_mode = "fixed_sbc_holdout"
    else:
        # Fallback: carve 10% from training pool (only if SBC not loaded)
        _n = len(_train_pool)
        _rng_tmp = torch.Generator().manual_seed(_SPLIT_SEED)
        _idx_tmp = torch.randperm(_n, generator=_rng_tmp).tolist()
        _n_test  = int(_n * (1 - _TRAIN_FRAC - _VAL_FRAC))
        _test_items   = [_train_pool[i] for i in _idx_tmp[-_n_test:]]
        _train_pool   = [_train_pool[i] for i in _idx_tmp[:-_n_test]]
        print(f"  ⚠  SBC not loaded — test is random {len(_test_items):,} samples (not comparable to baseline!)")
        _split_mode = "random_fallback"

    # ── 4. Train / val split ──────────────────────────────────────────────────
    _n_tv    = len(_train_pool)
    _n_train = int(_n_tv * (_TRAIN_FRAC / (_TRAIN_FRAC + _VAL_FRAC)))
    _rng     = torch.Generator().manual_seed(_SPLIT_SEED)
    _idx     = torch.randperm(_n_tv, generator=_rng).tolist()
    _train_items = [_train_pool[i] for i in _idx[:_n_train]]
    _val_items   = [_train_pool[i] for i in _idx[_n_train:]]
    print(f"  train={len(_train_items):,}  val={len(_val_items):,}  test={len(_test_items):,}")

    # ── 5. Build datasets ─────────────────────────────────────────────────────
    _tokenizer = DistilBertTokenizerFast.from_pretrained(_MODEL_NAME)
    _train_ds  = ProsodyDataset(_train_items, _tokenizer,
                                max_length=_MAX_LENGTH,
                                strip_punctuation=_STRIP_PUNCT,
                                extra_features=_EXTRA_FEATURES)
    _val_ds    = ProsodyDataset(_val_items,   _tokenizer,
                                max_length=_MAX_LENGTH,
                                strip_punctuation=_STRIP_PUNCT,
                                extra_features=_EXTRA_FEATURES)
    _test_ds   = ProsodyDataset(_test_items,  _tokenizer,
                                max_length=_MAX_LENGTH,
                                strip_punctuation=_STRIP_PUNCT,
                                extra_features=_EXTRA_FEATURES)
    print("✓ Datasets built.")

    _train_loader = DataLoader(_train_ds, batch_size=_BATCH_SIZE,
                               shuffle=True,  collate_fn=prosody_collate_fn)
    _val_loader   = DataLoader(_val_ds,   batch_size=_BATCH_SIZE,
                               shuffle=False, collate_fn=prosody_collate_fn)
    _test_loader  = DataLoader(_test_ds,  batch_size=_BATCH_SIZE,
                               shuffle=False, collate_fn=prosody_collate_fn)

    # ── 6. Model ──────────────────────────────────────────────────────────────
    _cfg_model = DistilBertConfig.from_pretrained(_MODEL_NAME,
                                                   num_labels=2,
                                                   seq_classif_dropout=0.2)
    _model = ProsodyBoundaryModel.from_pretrained(_MODEL_NAME,
                                                   config=_cfg_model,
                                                   ignore_mismatched_sizes=True,
                                                   _fast_init=False)
    _model.gradient_checkpointing_enable()
    _model = _model.to(device)
    print(f"✓ Model loaded ({sum(p.numel() for p in _model.parameters() if p.requires_grad):,} params)")

    # ── 7. Class-imbalance weights ────────────────────────────────────────────
    if _IMBALANCE == "weighted_loss":
        _loss_weights = torch.tensor([1.0, _BLW], dtype=torch.float).to(device)
        print(f"ℹ  Weighted loss  [non-boundary=1.0, boundary={_BLW}]")
    else:
        _loss_weights = None
        print("ℹ  No class-imbalance correction")

    # ── 8. Optimizer + scheduler ──────────────────────────────────────────────
    _optimizer   = AdamW(_model.parameters(), lr=_LR, weight_decay=_WD)
    _total_steps = _EPOCHS * len(_train_loader)
    _scheduler   = get_linear_schedule_with_warmup(
        _optimizer, num_warmup_steps=_WARMUP, num_training_steps=_total_steps)

    # ── 9. Training loop ──────────────────────────────────────────────────────
    _history = {
        "train_loss": [], "val_loss": [],
        "train_boundary_f1": [],  "val_boundary_f1": [],
        "train_boundary_prec": [], "val_boundary_prec": [],
        "train_boundary_rec": [],  "val_boundary_rec": [],
        "train_intonation_f1": [], "val_intonation_f1": [],
        "train_break_idx_f1": [],  "val_break_idx_f1": [],
    }
    _best_val_f1 = -1.0
    _best_epoch  = -1
    _best_state  = None

    print(f"\nTraining: {_EPOCHS} epochs | {len(_train_loader)} steps/epoch | {_total_steps} total steps")
    print("=" * 70)

    for epoch in tqdm_nb(range(1, _EPOCHS + 1), desc=f"{run_id[:30]} epochs"):
        # ── Train ──────────────────────────────────────────────────────────
        _model.train()
        _train_loss = 0.0
        t_b_logits, t_b_labels, t_b_spk_masks = [], [], []
        t_i_logits, t_i_labels = [], []
        t_x_logits, t_x_labels = [], []

        for batch in tqdm_nb(_train_loader, desc=f"Ep {epoch:02d} train", leave=False):
            _ids  = batch["input_ids"].to(device)
            _mask = batch["attention_mask"].to(device)
            _bl   = batch["labels"].to(device)
            _il   = batch["intonation_labels"].to(device)
            _xl   = batch["break_idx_labels"].to(device)

            _optimizer.zero_grad()
            out = _model(_ids, _mask, labels=_bl,
                         intonation_labels=_il, break_idx_labels=_xl,
                         loss_weights=_loss_weights)
            losses = out["losses"]
            loss = sum(
                w * losses[k]
                for k, w in [("boundary", _BTW), ("intonation", _ITW), ("break_idx", _XTW)]
                if k in losses
            )
            loss.backward()
            nn.utils.clip_grad_norm_(_model.parameters(), max_norm=1.0)
            _optimizer.step()
            _scheduler.step()

            _train_loss += loss.item()
            t_b_logits.append(out["boundary_logits"].detach().cpu())
            t_b_labels.append(_bl.detach().cpu())
            t_b_spk_masks.append(batch["spk_mask"].cpu())
            t_i_logits.append(out["intonation_logits"].detach().cpu())
            t_i_labels.append(_il.detach().cpu())
            t_x_logits.append(out["break_idx_logits"].detach().cpu())
            t_x_labels.append(_xl.detach().cpu())

        _train_loss /= len(_train_loader)
        t_bm = compute_metrics(*flatten_predictions(t_b_logits, t_b_labels, all_spk_masks=t_b_spk_masks))
        t_im = compute_multiclass_metrics(*flatten_predictions(t_i_logits, t_i_labels))
        t_xm = compute_multiclass_metrics(*flatten_predictions(t_x_logits, t_x_labels))

        # ── Validate ───────────────────────────────────────────────────────
        _model.eval()
        _val_loss = 0.0
        v_b_logits, v_b_labels, v_b_spk_masks = [], [], []
        v_i_logits, v_i_labels = [], []
        v_x_logits, v_x_labels = [], []

        with torch.no_grad():
            for batch in _val_loader:
                _ids  = batch["input_ids"].to(device)
                _mask = batch["attention_mask"].to(device)
                _bl   = batch["labels"].to(device)
                _il   = batch["intonation_labels"].to(device)
                _xl   = batch["break_idx_labels"].to(device)

                out = _model(_ids, _mask, labels=_bl,
                             intonation_labels=_il, break_idx_labels=_xl,
                             loss_weights=_loss_weights)
                losses = out["losses"]
                loss = sum(
                    w * losses[k]
                    for k, w in [("boundary", _BTW), ("intonation", _ITW), ("break_idx", _XTW)]
                    if k in losses
                )
                _val_loss += loss.item()
                v_b_logits.append(out["boundary_logits"].cpu())
                v_b_labels.append(_bl.cpu())
                v_b_spk_masks.append(batch["spk_mask"].cpu())
                v_i_logits.append(out["intonation_logits"].cpu())
                v_i_labels.append(_il.cpu())
                v_x_logits.append(out["break_idx_logits"].cpu())
                v_x_labels.append(_xl.cpu())

        _val_loss /= max(len(_val_loader), 1)
        v_bm = compute_metrics(*flatten_predictions(v_b_logits, v_b_labels, all_spk_masks=v_b_spk_masks))
        v_im = compute_multiclass_metrics(*flatten_predictions(v_i_logits, v_i_labels))
        v_xm = compute_multiclass_metrics(*flatten_predictions(v_x_logits, v_x_labels))

        _history["train_loss"].append(_train_loss)
        _history["val_loss"].append(_val_loss)
        _history["train_boundary_f1"].append(t_bm["f1"])
        _history["val_boundary_f1"].append(v_bm["f1"])
        _history["train_boundary_prec"].append(t_bm["precision"])
        _history["val_boundary_prec"].append(v_bm["precision"])
        _history["train_boundary_rec"].append(t_bm["recall"])
        _history["val_boundary_rec"].append(v_bm["recall"])
        _history["train_intonation_f1"].append(t_im["f1"])
        _history["val_intonation_f1"].append(v_im["f1"])
        _history["train_break_idx_f1"].append(t_xm["f1"])
        _history["val_break_idx_f1"].append(v_xm["f1"])

        star = "★" if v_bm["f1"] > _best_val_f1 else " "
        print(f"Ep {epoch:02d}/{_EPOCHS}  loss={_train_loss:.4f}→{_val_loss:.4f}  "
              f"| boundary F1={t_bm['f1']:.4f}→{v_bm['f1']:.4f} {star}"
              f"| inton F1={v_im['f1']:.4f}  break F1={v_xm['f1']:.4f}")

        if v_bm["f1"] > _best_val_f1:
            _best_val_f1 = v_bm["f1"]
            _best_epoch  = epoch
            _best_state  = {k: v.cpu().clone() for k, v in _model.state_dict().items()}

    print("=" * 70)
    print(f"Training complete.  Best val boundary F1 = {_best_val_f1:.4f}  (epoch {_best_epoch})")

    # ── 10. Test evaluation ───────────────────────────────────────────────────
    _model.load_state_dict(_best_state)
    _model.eval()

    tb_logits, tb_labels, tb_spk_masks = [], [], []
    ti_logits, ti_labels = [], []
    tx_logits, tx_labels = [], []
    per_sample = []

    with torch.no_grad():
        for batch in _test_loader:
            _ids  = batch["input_ids"].to(device)
            _mask = batch["attention_mask"].to(device)
            _bl   = batch["labels"].to(device)
            _il   = batch["intonation_labels"].to(device)
            _xl   = batch["break_idx_labels"].to(device)

            out     = _model(_ids, _mask)
            b_preds = out["boundary_logits"].argmax(dim=-1)
            i_preds = out["intonation_logits"].argmax(dim=-1)
            x_preds = out["break_idx_logits"].argmax(dim=-1)

            tb_logits.append(out["boundary_logits"]); tb_labels.append(_bl)
            tb_spk_masks.append(batch["spk_mask"])
            ti_logits.append(out["intonation_logits"]); ti_labels.append(_il)
            tx_logits.append(out["break_idx_logits"]); tx_labels.append(_xl)

            for sid, tokens, bp, bl, ip, il, xp, xl in zip(
                batch["sample_ids"], batch["tokens"],
                b_preds, _bl, i_preds, _il, x_preds, _xl,
            ):
                per_sample.append({
                    "sample_id":        sid,
                    "tokens":           tokens,
                    "boundary_preds":   bp[bl != -100].cpu().tolist(),
                    "boundary_true":    bl[bl != -100].cpu().tolist(),
                    "intonation_preds": ip[il != -100].cpu().tolist(),
                    "intonation_true":  il[il != -100].cpu().tolist(),
                    "break_idx_preds":  xp[xl != -100].cpu().tolist(),
                    "break_idx_true":   xl[xl != -100].cpu().tolist(),
                })

    fp_b, fl_b = flatten_predictions(tb_logits, tb_labels, all_spk_masks=tb_spk_masks)
    fp_i, fl_i = flatten_predictions(ti_logits, ti_labels)
    fp_x, fl_x = flatten_predictions(tx_logits, tx_labels)
    test_metrics = {
        "boundary":   compute_metrics(fp_b, fl_b),
        "intonation": compute_multiclass_metrics(fp_i, fl_i),
        "break_idx":  compute_multiclass_metrics(fp_x, fl_x),
    }

    print("=" * 55)
    print(f"  TEST SET  (best checkpoint — epoch {_best_epoch})")
    print(f"  Split mode: {_split_mode}")
    print("=" * 55)
    b = test_metrics["boundary"]
    print(f"  Boundary   F1={b['f1']:.4f}  P={b['precision']:.4f}  R={b['recall']:.4f}"
          f"  {'✓ beats baseline!' if b['f1'] > 0.77 else '✗ below 0.77 baseline'}")
    ii = test_metrics["intonation"]
    print(f"  Intonation F1={ii['f1']:.4f}  (macro, 3-class)")
    x = test_metrics["break_idx"]
    print(f"  Break idx  F1={x['f1']:.4f}  (macro, 2-class)")
    print("=" * 55)

    # ── 11. Plots ─────────────────────────────────────────────────────────────
    epochs_ax = list(range(1, _EPOCHS + 1))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    ax = axes[0]
    ax.plot(epochs_ax, _history["train_loss"], label="Train")
    ax.plot(epochs_ax, _history["val_loss"],   label="Val", linestyle="--")
    ax.axvline(_best_epoch, color="grey", linestyle=":", linewidth=1,
               label=f"Best (ep {_best_epoch})")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Combined loss")
    ax.set_title("Loss"); ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    ax.plot(epochs_ax, _history["train_boundary_f1"],  label="Train boundary F1")
    ax.plot(epochs_ax, _history["val_boundary_f1"],    label="Val boundary F1",   linestyle="--")
    ax.plot(epochs_ax, _history["val_intonation_f1"],  label="Val intonation F1", linestyle=":")
    ax.plot(epochs_ax, _history["val_break_idx_f1"],   label="Val break idx F1",  linestyle="-.")
    ax.axhline(0.77, color="red", linestyle=":", linewidth=1.2, label="Baseline F1 = 0.77")
    ax.axvline(_best_epoch, color="grey", linestyle=":", linewidth=1)
    ax.set_xlabel("Epoch"); ax.set_ylabel("F1")
    ax.set_title("F1 by head"); ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(0, 1)

    fig.suptitle(run_id, fontsize=8, color="grey")
    plt.tight_layout()
    plt.savefig(f"/tmp/{run_id}_curves.png", dpi=120, bbox_inches="tight")
    plt.show()

    cm = confusion_matrix(fl_b, fp_b, labels=[0, 1])
    fig2, ax2 = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=["non-boundary", "boundary"]).plot(
        ax=ax2, colorbar=False, cmap="Blues", values_format="d")
    ax2.set_title(f"Boundary — epoch {_best_epoch}\n{run_id}", fontsize=8)
    plt.tight_layout()
    plt.savefig(f"/tmp/{run_id}_confusion_matrix.png", dpi=120, bbox_inches="tight")
    plt.show()

    # ── 12. Save to Drive ─────────────────────────────────────────────────────
    os.makedirs(run_dir, exist_ok=True)
    ckpt_dir = os.path.join(run_dir, "checkpoint")
    _model.save_pretrained(ckpt_dir)
    _tokenizer.save_pretrained(ckpt_dir)

    hparam_record = {
        "run_id": run_id, "run_notes": _NOTES,
        "timestamp": datetime.now().isoformat(),
        "data": {
            "datasets": datasets,
            "n_samples_train_pool": len(_train_pool),
            "n_train": len(_train_ds), "n_val": len(_val_ds), "n_test": len(_test_ds),
            "split_mode": _split_mode,
            "split_seed": _SPLIT_SEED, "train_frac": _TRAIN_FRAC, "val_frac": _VAL_FRAC,
        },
        "preprocessing": {
            "strip_punctuation": _STRIP_PUNCT,
            "extra_features": _EXTRA_FEATURES, "max_length": _MAX_LENGTH,
        },
        "model": {"base": _MODEL_NAME},
        "training": {
            "batch_size": _BATCH_SIZE, "learning_rate": _LR,
            "num_epochs": _EPOCHS, "warmup_steps": _WARMUP, "weight_decay": _WD,
            "imbalance_strategy": _IMBALANCE, "boundary_loss_weight": _BLW,
            "boundary_task_weight": _BTW, "intonation_task_weight": _ITW,
            "break_idx_task_weight": _XTW, "best_epoch": _best_epoch,
        },
        "results": {
            "best_val_boundary_f1": _best_val_f1,
            "test": test_metrics,
            "val_history": {
                "loss":          _history["val_loss"],
                "boundary_f1":   _history["val_boundary_f1"],
                "boundary_prec": _history["val_boundary_prec"],
                "boundary_rec":  _history["val_boundary_rec"],
                "intonation_f1": _history["val_intonation_f1"],
                "break_idx_f1":  _history["val_break_idx_f1"],
            },
            "train_history": {
                "loss":          _history["train_loss"],
                "boundary_f1":   _history["train_boundary_f1"],
                "intonation_f1": _history["train_intonation_f1"],
                "break_idx_f1":  _history["train_break_idx_f1"],
            },
        },
    }
    with open(os.path.join(run_dir, f"{run_id}_hparams.json"), "w") as f:
        json.dump(hparam_record, f, indent=2, cls=_NumpyEncoder)
    with open(os.path.join(run_dir, f"{run_id}_test_predictions.json"), "w") as f:
        json.dump(per_sample, f, indent=2)
    for tag in ["curves", "confusion_matrix"]:
        shutil.copy(f"/tmp/{run_id}_{tag}.png", os.path.join(run_dir, f"{run_id}_{tag}.png"))

    print(f"\n✓ Saved → {run_dir}")

    # ── 13. Clear GPU memory ──────────────────────────────────────────────────
    _model.cpu()
    del _model, _optimizer, _scheduler, _best_state
    del _train_ds, _val_ds, _test_ds
    del _train_loader, _val_loader, _test_loader
    gc.collect()
    torch.cuda.empty_cache()
    subprocess.run(["sync"], check=False)
    print(f"✓ GPU memory cleared.  Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    return {"run_id": run_id, "run_dir": run_dir, "test_metrics": test_metrics}


print("✓ run_experiment() defined.")


---
## Section 14 · Run queue

Edit `RUNS` to define which datasets + overrides to run. Execute top-to-bottom after all prior cells.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 14 — Run queue                                         ║
# ║  Edit this cell to define which runs to execute.             ║
# ║                                                              ║
# ║  Each run specifies:                                         ║
# ║    "datasets"  : list of corpus keys from CORPUS_REGISTRY    ║
# ║    "overrides" : any Cell-1 config keys to override          ║
# ║                                                              ║
# ║  Valid dataset keys: "libri", "sbc", "bu"                    ║
# ║  (any key not in CORPUS_REGISTRY will raise a KeyError)      ║
# ╚══════════════════════════════════════════════════════════════╝

RUNS = [
    # ── LibriTTS + SBC + BU full combined, standard loss ─────────────────
    {
        "datasets": ["libri", "sbc", "bu"],
        "overrides": {
            "STRIP_PUNCTUATION": False,
            "IMBALANCE_STRATEGY": "none",
            "RUN_NOTES": "libri_sbc_bu_stl_pp",
        },
    },
        # ── SBC + BU gold-only, standard loss ────────────────────────────────
    {
        "datasets": ["sbc", "bu"],
        "overrides": {
            "STRIP_PUNCTUATION": False,
            "IMBALANCE_STRATEGY": "none",
            "RUN_NOTES": "sbc_bu_stl_pp",
        },
    },
    # ── LibriTTS + SBC + BU full combined, weighted loss ─────────────────
    {
        "datasets": ["libri", "sbc", "bu"],
        "overrides": {
            "STRIP_PUNCTUATION": False,
            "IMBALANCE_STRATEGY": "weighted_loss",
            "BOUNDARY_LOSS_WEIGHT": 4.0,
            "RUN_NOTES": "libri_sbc_bu_weighted_pp",
        },
    },
]

# ── Execute ───────────────────────────────────────────────────────────────────
results_summary = []
for i, run_def in enumerate(RUNS):
    print(f"\n{'='*70}\n  STARTING RUN {i+1} of {len(RUNS)}\n{'='*70}")
    result = run_experiment(
        overrides=run_def["overrides"],
        datasets=run_def["datasets"],
    )
    results_summary.append(result)

print("\n\n" + "█"*70)
print("  ALL RUNS COMPLETE — SUMMARY")
print("█"*70)
print(f"  {'Run ID':<45}  {'Boundary F1':>12}  {'Inton F1':>10}  {'Break F1':>10}")
print("  " + "-"*83)
for r in results_summary:
    b  = r["test_metrics"]["boundary"]["f1"]
    iv = r["test_metrics"]["intonation"]["f1"]
    x  = r["test_metrics"]["break_idx"]["f1"]
    flag = "✓" if b > 0.77 else "✗"
    print(f"  {r['run_id']:<45}  {b:>11.4f}{flag}  {iv:>10.4f}  {x:>10.4f}")
print("█"*70)
